# 路径规划一（TSP, Travelling Saleman Problem）

## 需求

> 给出N个位置点：0,1,2...,N-1。要求从0点出发，访问所有节点一次回到0点的最短路径。

### 方案一

> 最容易想到的是暴力破解，遍历所有的情况，时间复杂度是O(n!)，适合n非常小的场景

In [1]:
def tsp_brute_force(dist):
    from itertools import permutations

    n = len(dist)
    cities = list(range(n))
    min_cost = float('inf')
    min_tour = None

    for perm in permutations(cities[1:]):
        tour = [0] + list(perm) + [0]
        cost = sum(dist[tour[i]][tour[i+1]] for i in range(n))
        if cost < min_cost:
            min_cost = cost
            min_tour = tour

    return min_cost, min_tour

dist = [
    [0, 10, 20, 15],
    [10,  0, 12, 18],
    [20, 12,  0,  8],
    [15, 18,  8,  0]
]

cost, path = tsp_brute_force(dist)
city = ['A', 'B', 'C', 'D']
print(f"最短路程: {cost}")
print(f"最优路径: {' → '.join(city[i] for i in path)}")

最短路程: 45
最优路径: A → B → C → D → A


### 方案二

> 使用贪心算法，每次找距离当前位置最近的下一个点，时间复杂度是O(n^2), 结果不是最优解，适合对时间敏感但结果要求不高的场景

In [2]:
def tsp_greedy(dist):
    n = len(dist)
    visited = [False] * n
    tour = [0]
    visited[0] = True
    cost = 0

    for _ in range(1, n):
        last = tour[-1]
        next_cost, next_city= min((dist[last][j], j) for j in range(n) if not visited[j])
        tour.append(next_city)
        visited[next_city] = True
        cost += next_cost

    cost += dist[tour[-1]][0]  # Return to the starting city
    tour.append(0)  # Return to the starting city
    
    return cost, tour

dist = [
    [0, 10, 20, 15],
    [10,  0, 12, 18],
    [20, 12,  0,  8],
    [15, 18,  8,  0]
]

cost, path = tsp_greedy(dist)
city = ['A', 'B', 'C', 'D']
print(f"最短路程: {cost}")
print(f"最优路径: {' → '.join(city[i] for i in path)}")

最短路程: 45
最优路径: A → B → C → D → A


### 方案三

> 使用动态规划，时间、空间复杂度均为O(n^2·2^n)，适合n＜20的场景
> 1. 算出两两之间的距离dist, dist[i][j]表示i到j的距离
> 2. dp[mask][i]表示已经走过mask表示的节点，最后停留在i上的最短距离，mask是二进制掩码，例如1001，表示已经走过节点0和3，dp[1001][3]表示从节点0出发到节点3，当前停留在节点三上
> 3. 状态转移方程：dp[mask][j] = min(dp[mask^(1 << j)][i] + dist[i][j]), i≠j 且 i∈mask

In [3]:
def tsp_dp(dist):
    n = len(dist)
    INF = float('inf')
    FULL = (1 << n) - 1
    
    dp = [[INF] * n for _ in range(1 << n)]
    dp[1 << 0][0] = 0
    path = [[-1] * n for _ in range(1 << n)]

    for mask in range(1, 1 << n):
        if not (mask & 1): # start from node0
            continue
        
        for i in range(n):
            if not (mask & (1 <<i)): # i must in mask
                continue
            if dp[mask][i] == INF:
                continue
            for j in range(n):
                if mask & (1 << j):
                    continue
                new_mask = mask | (1 << j)
                new_cost = dp[mask][i] + dist[i][j]
                if new_cost < dp[new_mask][j]:
                    dp[new_mask][j] = new_cost
                    path[new_mask][j] = i
    
    best_cost = INF
    best_last = -1

    for k in range(1, n):
        if dp[FULL][k] == INF:
            continue
        cost = dp[FULL][k] + dist[k][0]
        if cost < best_cost:
            best_cost = cost
            best_last = k
    
    best_path = []
    cur = best_last
    mask = FULL
    while cur != -1:
        best_path.append(cur)
        prev = path[mask][cur]
        mask ^= (1 << cur)
        cur = prev
    best_path.reverse()
    best_path.append(0)

    return best_cost, best_path

        
dist = [
    [0, 10, 20, 15],
    [10,  0, 12, 18],
    [20, 12,  0,  8],
    [15, 18,  8,  0]
]

cost, path = tsp_dp(dist)
city = ['A', 'B', 'C', 'D']
print(f"最短路程: {cost}")
print(f"最优路径: {' → '.join(city[i] for i in path)}")

最短路程: 45
最优路径: A → D → C → B → A


### 方案四

> 贪心算法优点是快，缺点是只能求局部最优解；DP正好反过来。有没有一种算法，能综合他们的优点：在短时间内求得近似最优解。
>
> 模拟退火算法
> 退火是一种金属或热材料处理工艺，指将材料加热到一定的温度，然后缓慢冷却的工艺。
> 模拟退火算法的核心思想：高温时分子能量高，容易乱跑（算法跳出局部最优解的概率高），随着温度下降，分子活动变低，最后趋于稳定状态（算法收敛到近似最优解）
>
> 核心接受概率: $$ P = e^{-\frac{\Delta}{T}} if \Delta \lt 0 else 1.0 $$ 温度T越高，接受坏结果的可能性越高
>
> 复杂度：O(iter * n), iter 》 n

In [17]:
"""
TSP 模拟退火求解器 —— 完整最终版
优化点：
  1. 贪心初始解（最近邻）
  2. 2-opt 增量计算 O(1) delta
  3. 自适应降温
  4. 限制邻域范围（与温度联动）
  5. 扰动重启 double-bridge（代替简单 restart）
"""

import math
import random
import time
from typing import Optional


# ─────────────────────────────────────────────
# 工具函数
# ─────────────────────────────────────────────

def total_cost(path: list[int], dist: list[list[float]]) -> float:
    """计算路径总距离（仅用于初始化和最终验证）"""
    n = len(path)
    return sum(dist[path[i]][path[(i + 1) % n]] for i in range(n))


def greedy_init(dist: list[list[float]], start: int = 0) -> list[int]:
    """
    贪心最近邻初始解，固定从 start 城市出发。
    时间复杂度 O(n²)
    """
    n = len(dist)
    visited = [False] * n
    path = [start]
    visited[start] = True
    unvisited = set(range(n))
    unvisited.remove(start)

    while unvisited:
        last = path[-1]
        nxt = min(unvisited, key=lambda j: dist[last][j])
        path.append(nxt)
        visited[nxt] = True
        unvisited.remove(nxt)

    return path


def double_bridge(path: list[int]) -> list[int]:
    """
    double-bridge 扰动：将路径切成 4 段后重新拼接，
    产生 2-opt 无法到达的邻域，用于跳出局部最优。

    原路径：A - B - C - D
    扰动后：A - C - B - D
    """
    n = len(path)
    # 随机选 3 个切割点，保证每段至少 1 个城市
    cuts = sorted(random.sample(range(1, n), 3))
    a, b, c = cuts
    # 重新拼接：段 A + 段 C + 段 B + 段 D
    new_path = path[:a] + path[c:] + path[b:c] + path[a:b]
    return new_path


# ─────────────────────────────────────────────
# 核心 SA 求解
# ─────────────────────────────────────────────

def sa_tsp(
    dist: list[list[float]],
    start: int = 0,
    T_init: float = None,
    T_min: float = 1e-4,
    cooling: float = 0.9995,
    max_iter: int = 200_000,
    adaptive: bool = True,
    seed: Optional[int] = None,
) -> tuple[float, list[int]]:
    """
    单次模拟退火求解 TSP。

    参数
    ----
    dist       : n×n 距离矩阵
    start      : 起始城市编号（固定）
    T_init     : 初始温度，None 则自动估算
    T_min      : 终止温度
    cooling    : 冷却率（每步 T *= cooling）
    max_iter   : 最大迭代次数
    adaptive   : 是否启用自适应降温
    seed       : 随机种子（可复现）

    返回
    ----
    (best_cost, best_path)
    """
    if seed is not None:
        random.seed(seed)

    n = len(dist)
    path = greedy_init(dist, start)
    cur_cost = total_cost(path, dist)
    best_path = path[:]
    best_cost = cur_cost

    # 自动估算初始温度：令初始接受率约 80%
    # 抽样若干随机邻域的 delta，取平均值反推 T
    if T_init is None:
        deltas = []
        for _ in range(200):
            i = random.randint(1, n - 2)
            j = random.randint(i + 1, n - 1)
            a, b = path[i - 1], path[i]
            c, d = path[j], path[(j + 1) % n]
            d_val = (dist[a][c] + dist[b][d]) - (dist[a][b] + dist[c][d])
            if d_val > 0:
                deltas.append(d_val)
        avg_delta = sum(deltas) / len(deltas) if deltas else cur_cost * 0.01
        T_init = -avg_delta / math.log(0.8)  # P = e^(-avg_delta/T) = 0.8

    T = T_init

    # 自适应降温：每 500 步统计接受率
    window = 500
    accept_count = 0

    for step in range(max_iter):
        if T < T_min:
            break

        # 邻域生成：2-opt，限制反转长度与温度联动
        # 高温允许大范围反转，低温限制小范围
        max_flip = max(2, int((n - 1) * (T / T_init) ** 0.3))
        i = random.randint(1, n - 2)
        j = min(i + random.randint(1, max_flip), n - 1)

        # 增量计算 delta（O(1)）
        a = path[i - 1]
        b = path[i]
        c = path[j]
        d = path[(j + 1) % n]
        delta = (dist[a][c] + dist[b][d]) - (dist[a][b] + dist[c][d])

        # Metropolis 接受准则
        if delta < 0 or random.random() < math.exp(-delta / T):
            path[i: j + 1] = path[i: j + 1][::-1]
            cur_cost += delta
            accept_count += 1

            if cur_cost < best_cost:
                best_cost = cur_cost
                best_path = path[:]

        # 固定冷却
        T *= cooling

        # 自适应修正（每 window 步）
        if adaptive and (step + 1) % window == 0:
            accept_rate = accept_count / window
            if accept_rate < 0.01:
                # 接受率过低 → 小幅回温
                T = min(T * 1.5, T_init * 0.5)
            elif accept_rate > 0.5 and T > T_init * 0.01:
                # 接受率过高 → 适当加速冷却
                T *= 0.98
            accept_count = 0

    return best_cost, best_path


# ─────────────────────────────────────────────
# 扰动重启（代替简单多次重启）
# ─────────────────────────────────────────────

def sa_tsp_with_perturbation(
    dist: list[list[float]],
    start: int = 0,
    n_perturb: int = 5,
    max_iter_main: int = 200_000,
    max_iter_rerun: int = 50_000,
    **sa_kwargs,
) -> tuple[float, list[int]]:
    """
    主 SA + 扰动重启策略：
      1. 跑一次完整 SA 得到初始最优解
      2. 对最优解做 double-bridge 扰动
      3. 从扰动解出发再跑短 SA
      4. 重复 n_perturb 次，保留全局最优

    比单纯多次随机重启更高效：保留了已有好结构。

    参数
    ----
    n_perturb       : 扰动重启次数
    max_iter_main   : 第一次 SA 的迭代次数
    max_iter_rerun  : 每次扰动后 SA 的迭代次数
    """
    # 第一次完整 SA
    best_cost, best_path = sa_tsp(
        dist, start=start, max_iter=max_iter_main, **sa_kwargs
    )

    for _ in range(n_perturb):
        # 扰动：double-bridge 打乱部分结构
        perturbed = double_bridge(best_path)

        # 确保起始城市不变（把 start 移到首位）
        if perturbed[0] != start:
            idx = perturbed.index(start)
            perturbed = perturbed[idx:] + perturbed[:idx]

        # 从扰动解出发，跑短 SA
        # 注意：直接设置初始路径绕过贪心，需小幅调整接口
        cost, path = _sa_from_init(
            dist, perturbed, max_iter=max_iter_rerun, **sa_kwargs
        )

        if cost < best_cost:
            best_cost, best_path = cost, path

    return best_cost, best_path


def _sa_from_init(
    dist: list[list[float]],
    init_path: list[int],
    T_init: float = None,
    T_min: float = 1e-4,
    cooling: float = 0.9998,
    max_iter: int = 50_000,
    adaptive: bool = True,
    **_,
) -> tuple[float, list[int]]:
    """从指定初始路径开始跑 SA（内部使用）"""
    n = len(dist)
    path = init_path[:]
    cur_cost = total_cost(path, dist)
    best_path = path[:]
    best_cost = cur_cost

    if T_init is None:
        T_init = cur_cost * 0.005
    T = T_init

    window = 500
    accept_count = 0

    for step in range(max_iter):
        if T < T_min:
            break

        max_flip = max(2, int((n - 1) * (T / T_init) ** 0.3))
        i = random.randint(1, n - 2)
        j = min(i + random.randint(1, max_flip), n - 1)

        a, b = path[i - 1], path[i]
        c, d = path[j], path[(j + 1) % n]
        delta = (dist[a][c] + dist[b][d]) - (dist[a][b] + dist[c][d])

        if delta < 0 or random.random() < math.exp(-delta / T):
            path[i: j + 1] = path[i: j + 1][::-1]
            cur_cost += delta
            accept_count += 1
            if cur_cost < best_cost:
                best_cost = cur_cost
                best_path = path[:]

        T *= cooling

        if adaptive and (step + 1) % window == 0:
            rate = accept_count / window
            if rate < 0.01:
                T = min(T * 1.5, T_init * 0.5)
            accept_count = 0

    return best_cost, best_path


# ─────────────────────────────────────────────
# 使用示例
# ─────────────────────────────────────────────

if __name__ == "__main__":
    import random as _rnd

    # 生成随机测试距离矩阵（对称）
    # _rnd.seed(42)
    N = 30
    coords = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(N)]

    def euclidean(a, b):
        return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

    dist_matrix = [
        [euclidean(coords[i], coords[j]) for j in range(N)]
        for i in range(N)
    ]

    print(f"城市数量: {N}")
    print(f"贪心初始解: {total_cost(greedy_init(dist_matrix), dist_matrix):.2f}")

    # 方案A：单次 SA（速度快）
    t0 = time.time()
    cost_a, path_a = sa_tsp(dist_matrix, start=0, max_iter=200_000)
    print(f"\n[单次 SA]  距离: {cost_a:.2f}  耗时: {time.time()-t0:.2f}s")
    print(f"路径: {path_a}")

    # 方案B：SA + 扰动重启（质量更稳定）
    t0 = time.time()
    cost_b, path_b = sa_tsp_with_perturbation(
        dist_matrix,
        start=0,
        n_perturb=5,
        max_iter_main=150_000,
        max_iter_rerun=30_000,
    )
    print(f"\n[SA+扰动]  距离: {cost_b:.2f}  耗时: {time.time()-t0:.2f}s")
    print(f"路径: {path_b}")

城市数量: 30
贪心初始解: 528.58



[单次 SA]  距离: 499.60  耗时: 0.50s
路径: [0, 1, 8, 11, 15, 5, 22, 25, 19, 26, 10, 9, 12, 14, 3, 16, 13, 29, 2, 20, 4, 21, 18, 28, 23, 6, 7, 24, 27, 17]

[SA+扰动]  距离: 415.23  耗时: 0.89s
路径: [0, 28, 18, 21, 4, 20, 2, 29, 13, 16, 3, 14, 12, 9, 10, 26, 27, 19, 24, 25, 22, 5, 15, 11, 8, 1, 6, 7, 17, 23]


### 方案五

> 遗传算法，和方案四的想法差不多
> 一条路径代表一个个体，多条路径组成一个族群
> 每次进化从族群中选出两个父代，经过交叉、变异后生成子代，子代进行2-opt进化
> 最后从中选取最优的进化解

In [2]:
"""
TSP 遗传算法求解器 —— 完整改进版
改进点：
  1. 锦标赛选择（替代轮盘赌，对极端值不敏感）
  2. 贪心初始解（最近邻）+ 随机混合种群
  3. OX 顺序交叉（保证城市不重复）
  4. 混合变异（swap + 小范围 2-opt）
  5. 多精英保留
  6. Memetic：每个子代跑 2-opt 局部搜索（增量 delta，O(n²) 收敛）
  7. 输出路径统一从城市 0 开始
"""

import math
import random
import time
from typing import Optional


# ─────────────────────────────────────────────
# 工具函数
# ─────────────────────────────────────────────

def total_cost(path: list[int], dist: list[list[float]]) -> float:
    """计算路径总距离（环形）"""
    n = len(path)
    return sum(dist[path[i]][path[(i + 1) % n]] for i in range(n))


def normalize_path(path: list[int], start: int = 0) -> list[int]:
    """将路径旋转，使其从 start 城市开始"""
    idx = path.index(start)
    return path[idx:] + path[:idx]


def greedy_init(dist: list[list[float]], start: int = 0) -> list[int]:
    """
    贪心最近邻初始解，从 start 出发。
    时间复杂度 O(n²)
    """
    n = len(dist)
    path = [start]
    unvisited = set(range(n))
    unvisited.remove(start)

    while unvisited:
        last = path[-1]
        nxt = min(unvisited, key=lambda j: dist[last][j])
        path.append(nxt)
        unvisited.remove(nxt)

    return path


# ─────────────────────────────────────────────
# 2-opt 局部搜索（增量计算）
# ─────────────────────────────────────────────

def two_opt(path: list[int], dist: list[list[float]]) -> tuple[list[int], float]:
    """
    2-opt 局部搜索，反复扫描直到无法改进。
    使用增量 delta 计算，内层 O(1)，整体 O(n²) 每轮。

    返回 (优化后路径, 优化后 cost)
    """
    n = len(path)
    best = path[:]
    best_cost = total_cost(best, dist)
    improved = True

    while improved:
        improved = False
        for i in range(n - 1):
            for j in range(i + 2, n):
                # 跳过首尾相连的情况（等价于不做任何操作）
                if i == 0 and j == n - 1:
                    continue

                # 增量计算：
                # 删除边 path[i-1]→path[i] 和 path[j]→path[j+1]
                # 添加边 path[i-1]→path[j] 和 path[i]→path[j+1]
                a = best[i - 1] if i > 0 else best[n - 1]
                b = best[i]
                c = best[j]
                d = best[(j + 1) % n]

                delta = (
                    dist[a][c] + dist[b][d]
                    - dist[a][b] - dist[c][d]
                )

                if delta < -1e-10:  # 用小 epsilon 避免浮点误差反复触发
                    best[i:j + 1] = best[i:j + 1][::-1]
                    best_cost += delta
                    improved = True

    return best, best_cost


# ─────────────────────────────────────────────
# 选择
# ─────────────────────────────────────────────

def tournament_select(
    pop: list[list[int]],
    costs: list[float],
    k: int = 3,
) -> list[int]:
    """
    锦标赛选择：随机抽 k 个个体，返回其中 cost 最小的。
    k 控制选择压力：k 越大越偏向最优个体。
    """
    indices = random.sample(range(len(pop)), k)
    best_idx = min(indices, key=lambda i: costs[i])
    return pop[best_idx]


# ─────────────────────────────────────────────
# 交叉
# ─────────────────────────────────────────────

def ox_crossover(parent_a: list[int], parent_b: list[int]) -> list[int]:
    """
    顺序交叉 OX：
      1. 从 parent_a 复制随机片段 [i, j)
      2. 剩余位置按 parent_b 的顺序填入未出现的城市
    保证子代每个城市恰好出现一次。
    """
    n = len(parent_a)
    i, j = sorted(random.sample(range(n), 2))

    child = [-1] * n
    child[i:j] = parent_a[i:j]

    # parent_b 中未被复制的城市，保留其相对顺序
    in_child = set(child[i:j])
    remaining = [x for x in parent_b if x not in in_child]

    k = 0
    for idx in range(n):
        if child[idx] == -1:
            child[idx] = remaining[k]
            k += 1

    return child


# ─────────────────────────────────────────────
# 变异
# ─────────────────────────────────────────────

def mutate(
    path: list[int],
    mutation_rate: float,
    max_2opt_range: int = 10,
) -> list[int]:
    """
    混合变异：
      - 50% 概率：swap（交换两个随机城市）→ 大扰动，保持多样性
      - 50% 概率：小范围 2-opt 反转 → 小扰动，精细调整
    """
    if random.random() >= mutation_rate:
        return path

    n = len(path)
    if random.random() < 0.5:
        # swap 变异
        i, j = random.sample(range(n), 2)
        path[i], path[j] = path[j], path[i]
    else:
        # 小范围 2-opt 变异（限制反转长度，避免破坏大段结构）
        i = random.randint(0, n - 3)
        j = min(i + random.randint(2, max_2opt_range), n - 1)
        path[i:j + 1] = path[i:j + 1][::-1]

    return path


# ─────────────────────────────────────────────
# 主算法
# ─────────────────────────────────────────────

def ga_tsp(
    dist: list[list[float]],
    pop_size: int = 100,
    generations: int = 300,
    mutation_rate: float = 0.2,
    elite_size: int = 5,
    tournament_k: int = 3,
    local_search: bool = True,
    start: int = 0,
    seed: Optional[int] = None,
    verbose: bool = True,
    verbose_interval: int = 50,
) -> tuple[float, list[int]]:
    """
    遗传算法求解 TSP。

    参数
    ----
    dist             : n×n 距离矩阵
    pop_size         : 种群大小
    generations      : 进化代数
    mutation_rate    : 变异率（建议 0.1~0.2）
    elite_size       : 每代直接保留的精英数量
    tournament_k     : 锦标赛选择的候选数（选择压力）
    local_search     : 是否对每个子代做 2-opt 局部搜索（Memetic）
    start            : 输出路径的起始城市
    seed             : 随机种子
    verbose          : 是否打印进化过程
    verbose_interval : 每隔多少代打印一次

    返回
    ----
    (best_cost, best_path)，路径从 start 城市开始
    """
    if seed is not None:
        random.seed(seed)

    n = len(dist)

    # ── 初始化种群：1 个贪心解 + 其余随机 ──
    def init_population() -> list[list[int]]:
        pop = [greedy_init(dist, start=0)]   # 贪心解固定从 0 出发
        while len(pop) < pop_size:
            p = list(range(n))
            random.shuffle(p)
            pop.append(p)
        return pop

    pop = init_population()

    # 计算初始种群的 cost（避免重复计算）
    costs = [total_cost(p, dist) for p in pop]

    best_idx = min(range(len(pop)), key=lambda i: costs[i])
    best_path = pop[best_idx][:]
    best_cost = costs[best_idx]

    if verbose:
        print(f"初始最优（贪心）: {best_cost:.2f}")

    # ── 主循环 ──
    for gen in range(generations):

        # 按 cost 排序（精英保留用）
        sorted_indices = sorted(range(len(pop)), key=lambda i: costs[i])
        new_pop = [pop[i][:] for i in sorted_indices[:elite_size]]
        new_costs = [costs[i] for i in sorted_indices[:elite_size]]

        # 生成子代
        while len(new_pop) < pop_size:
            parent1 = tournament_select(pop, costs, k=tournament_k)
            parent2 = tournament_select(pop, costs, k=tournament_k)

            child = ox_crossover(parent1, parent2)
            child = mutate(child, mutation_rate)

            # Memetic：子代做 2-opt 局部搜索
            if local_search:
                child, child_cost = two_opt(child, dist)
            else:
                child_cost = total_cost(child, dist)

            new_pop.append(child)
            new_costs.append(child_cost)

            # 顺手更新全局最优
            if child_cost < best_cost:
                best_cost = child_cost
                best_path = child[:]

        pop = new_pop
        costs = new_costs

        if verbose and (gen + 1) % verbose_interval == 0:
            avg_cost = sum(costs) / len(costs)
            print(f"Gen {gen + 1:4d} | best: {best_cost:.2f} | avg: {avg_cost:.2f}")

    # 输出路径统一从 start 城市开始
    best_path = normalize_path(best_path, start=start)

    if verbose:
        print(f"\n最终最优: {best_cost:.2f}")
        print(f"路径: {best_path}")

    return best_cost, best_path


# ─────────────────────────────────────────────
# 使用示例
# ─────────────────────────────────────────────

if __name__ == "__main__":
    # 生成随机测试距离矩阵（对称欧氏距离）
    # random.seed(42)
    N = 30
    coords = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(N)]

    def euclidean(a, b):
        return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)

    dist_matrix = [
        [euclidean(coords[i], coords[j]) for j in range(N)]
        for i in range(N)
    ]

    print(f"城市数量: {N}\n{'─'*40}")

    # 方案A：不带局部搜索（纯 GA，速度快）
    t0 = time.time()
    cost_a, path_a = ga_tsp(
        dist_matrix,
        pop_size=100,
        generations=300,
        local_search=False,
        verbose=True,
        verbose_interval=100,
    )
    print(f"耗时: {time.time() - t0:.2f}s\n{'─'*40}")

    # 方案B：带局部搜索（Memetic GA，质量更好）
    t0 = time.time()
    cost_b, path_b = ga_tsp(
        dist_matrix,
        pop_size=80,
        generations=150,       # 代数可以少，因为每代质量更高
        local_search=True,
        verbose=True,
        verbose_interval=50,
    )
    print(f"耗时: {time.time() - t0:.2f}s")

城市数量: 30
────────────────────────────────────────
初始最优（贪心）: 536.29
Gen  100 | best: 497.54 | avg: 519.52


Gen  200 | best: 492.54 | avg: 518.05
Gen  300 | best: 492.54 | avg: 511.74

最终最优: 492.54
路径: [0, 17, 22, 8, 13, 6, 15, 9, 19, 23, 25, 20, 27, 10, 11, 14, 21, 16, 12, 18, 1, 3, 24, 28, 26, 29, 7, 4, 5, 2]
耗时: 0.89s
────────────────────────────────────────
初始最优（贪心）: 536.29
Gen   50 | best: 475.98 | avg: 490.33
Gen  100 | best: 475.98 | avg: 488.63
Gen  150 | best: 475.98 | avg: 486.49

最终最优: 475.98
路径: [0, 17, 22, 8, 13, 20, 27, 10, 11, 14, 21, 25, 6, 15, 9, 19, 23, 16, 12, 18, 1, 3, 24, 28, 26, 29, 7, 4, 5, 2]
耗时: 2.22s
